# Collect Cosmos-Policy contrastive input pairs — *first-J inferences, index-paired*

A simpler variant of `collect_policy_inputs_milk.ipynb`. Instead of replaying
the cluttered-scene action sequence inside the uncluttered scene (or vice
versa) to force-match proprio per inference, this notebook runs **independent
rollouts** in the cluttered and uncluttered scenes under the same prompt and
the same `init_state` per episode, and captures the *first `J` inferences* of
each rollout.

Pairing is purely by index:

- `I(u, i, j)` = policy input on the `j`-th inference of the `i`-th episode
  rollout in the uncluttered (milk_only) scene `env_pos`.
- `I(c, i, j)` = policy input on the `j`-th inference of the `i`-th episode
  rollout in the cluttered (full 8-object) scene `env_neg`.
- Output rows are `(I(u, i, j), I(c, i, j))` for `j < J_THRESHOLD` and
  `i < N_EPISODES`, in order.

Caveat vs. the original notebook: row `i` in `positive.npz` and row `i` in
`negative.npz` do **not** share proprio. The (pos − neg) contrast therefore
mixes (visual distractor effect) with (whatever proprio divergence the two
policies have accumulated by step `j`). Downstream SVD code treats every
pair as an independent contrastive sample, so this still produces a valid
contrastive direction — it just integrates over the early-rollout proprio
distribution under each scene rather than being a strict per-pose matched
comparison.

Prompt (single, used everywhere):

  `"put both the milk and the tomato sauce in the basket"`

Output layout:

```
notebooks/lqr/inputs/policy_inputs/libero_10__task00__milk_first_chunks/
  positive.npz   # uncluttered renders (I(u, i, j)), first J inferences per ep
  negative.npz   # cluttered   renders (I(c, i, j)), first J inferences per ep
  manifest.json  # prompt, stress configs, per-rollout summaries
  derived__<slug>.bddl   # BDDL emitted by SceneRemoveObjects
```


In [ ]:
# notebooks/_setup.py lives two dirs up (notebooks/lqr/inputs/<this>.ipynb).
import sys; sys.path.insert(0, '../..')
from _setup import setup_env
setup_env()

import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')


In [ ]:
import copy
import hashlib
import json
import re
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple

import numpy as np

from libero.libero import benchmark, get_libero_path
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig, prepare_observation, TASK_MAX_STEPS,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action, get_model, load_dataset_stats, init_t5_text_embeddings_cache,
)


## 1. `SceneRemoveObjects` (unchanged from the original milk notebook)

Strips a list of object keys from the BDDL, rewrites the `:goal` so success
still scores against the surviving targets, and truncates LIBERO's saved
init_state to match the smaller body list. Used here only to build `env_pos`
(milk-only stripped scene) and to slice `init_state[ep]` for `env_pos`'s
smaller body layout. We do **not** need the inverse `_expand_pos_state_to_neg`
helper because each env runs independently — no cross-env state injection.


In [ ]:
# ---------- BDDL editing helpers ----------

def _find_section_bounds(bddl, keyword):
    """Return (start, end) of the `(:<keyword> ...)` block, paren-balanced, or None."""
    needle = f'(:{keyword}'
    start = bddl.find(needle)
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(bddl)):
        c = bddl[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                return start, i + 1
    return None


def _strip_paren_block(text, opening_token):
    """Find the first '(<opening_token>' and remove from there through the matching ')'
    plus trailing whitespace + newline. Returns text unchanged if not found."""
    idx = text.find(opening_token)
    if idx == -1:
        return text
    depth = 0
    end = None
    for i in range(idx, len(text)):
        c = text[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end is None:
        return text
    while end < len(text) and text[end] in ' \t':
        end += 1
    if end < len(text) and text[end] == '\n':
        end += 1
    line_start = text.rfind('\n', 0, idx) + 1
    if text[line_start:idx].strip() == '':
        idx = line_start
    return text[:idx] + text[end:]


def _drop_section_lines(bddl, keyword, predicate):
    """Within (:<keyword> ...) block, drop any line where predicate(stripped_line) is True."""
    bounds = _find_section_bounds(bddl, keyword)
    if bounds is None:
        return bddl
    s, e = bounds
    section = bddl[s:e]
    keep = []
    for line in section.split('\n'):
        if predicate(line.strip()):
            continue
        keep.append(line)
    return bddl[:s] + '\n'.join(keep) + bddl[e:]


def _parse_objects_order(bddl):
    """Return the list of object keys in :objects declaration order."""
    bounds = _find_section_bounds(bddl, 'objects')
    if bounds is None:
        raise ValueError('BDDL missing :objects section')
    s, e = bounds
    section = bddl[s:e]
    keys = []
    for line in section.split('\n'):
        stripped = line.strip()
        if not stripped or stripped.startswith('(:') or stripped == ')':
            continue
        m = re.match(r'(\S+)\s*-\s*\S+', stripped)
        if m:
            keys.append(m.group(1))
    return keys


def _resolve_libero_problem_obj(env):
    """Walk env.env.env... up to depth 8 looking for the LIBERO problem object."""
    cur, seen = env, set()
    for _ in range(8):
        if cur is None or id(cur) in seen:
            break
        seen.add(id(cur))
        if hasattr(cur, 'objects_dict') and hasattr(cur, 'fixtures_dict'):
            return cur
        cur = getattr(cur, 'env', None)
    return None


# ---------- the stress test ----------

class StressTest:
    """No-op baseline. Subclass and override hooks."""
    slug: str = 'stock'
    def transform_task(self, task, output_dir=None): return task
    def set_init_state(self, env, init_state, episode_idx=0): return env.set_init_state(init_state)
    def apply_to_env(self, env, episode_idx=0) -> None: return None
    def transform_task_desc(self, desc, env=None): return desc
    def manifest(self) -> dict: return {'kind': type(self).__name__, 'slug': self.slug}


@dataclass
class SceneRemoveObjects(StressTest):
    """Strip listed object keys from the BDDL, rewrite the :goal, rewrite the prompt."""
    remove: Tuple[str, ...] = ()
    goal_substitutions: Tuple[Tuple[str, str], ...] = ()
    prompt_replace_mapping: Tuple[Tuple[str, str], ...] = ()
    name_hint: str = 'remove'
    _bddl_path: Optional[str] = field(default=None, init=False, repr=False)
    _orig_order: Tuple[str, ...] = field(default=(), init=False, repr=False)
    _kept_order: Tuple[str, ...] = field(default=(), init=False, repr=False)

    @property
    def slug(self) -> str:
        payload = '|'.join(sorted(self.remove))
        payload += '||' + '|'.join(f'{a}=>{b}' for a, b in self.goal_substitutions)
        payload += '||' + '|'.join(f'{a}=>{b}' for a, b in self.prompt_replace_mapping)
        h = hashlib.md5(payload.encode()).hexdigest()[:6]
        return f'{self.name_hint}_n{len(self.remove)}_{h}'

    def transform_task(self, task, output_dir=None):
        if not self.remove and not self.goal_substitutions:
            return task
        src_path = os.path.join(
            get_libero_path('bddl_files'), task.problem_folder, task.bddl_file,
        )
        with open(src_path) as f:
            bddl = f.read()

        self._orig_order = tuple(_parse_objects_order(bddl))
        self._kept_order = tuple(k for k in self._orig_order if k not in self.remove)
        missing = set(self.remove) - set(self._orig_order)
        if missing:
            raise ValueError(f'remove keys not in :objects: {sorted(missing)}')

        bddl = _drop_section_lines(bddl, 'objects',
            lambda L: any(re.match(rf'{re.escape(k)}\s*-', L) for k in self.remove))
        bddl = _drop_section_lines(bddl, 'obj_of_interest',
            lambda L: L in self.remove)
        bddl = _drop_section_lines(bddl, 'init',
            lambda L: any(L.startswith(f'(On {k} ') for k in self.remove))

        bounds = _find_section_bounds(bddl, 'regions')
        if bounds is not None:
            s, e = bounds
            section = bddl[s:e]
            for k in self.remove:
                base = re.sub(r'_\d+$', '', k)
                section = _strip_paren_block(section, f'({base}_init_region')
            bddl = bddl[:s] + section + bddl[e:]

        if self.goal_substitutions:
            bounds = _find_section_bounds(bddl, 'goal')
            if bounds is not None:
                s, e = bounds
                section = bddl[s:e]
                for old, new in self.goal_substitutions:
                    section = section.replace(old, new)
                bddl = bddl[:s] + section + bddl[e:]

        out_dir = Path(output_dir) if output_dir else Path('/tmp')
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'derived__{self.slug}.bddl'
        out_path.write_text(bddl)
        self._bddl_path = str(out_path.resolve())

        if hasattr(task, '_replace'):
            return task._replace(bddl_file=self._bddl_path, problem_folder='')
        new_task = copy.copy(task)
        new_task.bddl_file = self._bddl_path
        new_task.problem_folder = ''
        return new_task

    def set_init_state(self, env, init_state, episode_idx=0):
        if not self.remove:
            return env.set_init_state(init_state)
        problem = _resolve_libero_problem_obj(env)
        if problem is None:
            return env.set_init_state(init_state)
        sim = problem.sim
        new_nq = int(sim.model.nq)
        new_nv = int(sim.model.nv)
        n_kept = len(self._kept_order)
        n_orig = len(self._orig_order)
        robot_nq = new_nq - 7 * n_kept
        robot_nv = new_nv - 6 * n_kept
        old_nq = robot_nq + 7 * n_orig
        old_nv = robot_nv + 6 * n_orig
        expected_len = 1 + old_nq + old_nv
        if len(init_state) != expected_len:
            raise ValueError(
                f'SceneRemoveObjects.set_init_state: saved init_state size '
                f'{len(init_state)} != expected {expected_len} (robot_nq={robot_nq}, '
                f'robot_nv={robot_nv}, n_orig={n_orig}). Did the BDDL change?'
            )

        qpos = list(init_state[1 : 1 + robot_nq])
        for k in self._kept_order:
            i = self._orig_order.index(k)
            start = 1 + robot_nq + 7 * i
            qpos.extend(init_state[start : start + 7])

        qvel = list(init_state[1 + old_nq : 1 + old_nq + robot_nv])
        for k in self._kept_order:
            i = self._orig_order.index(k)
            start = 1 + old_nq + robot_nv + 6 * i
            qvel.extend(init_state[start : start + 6])

        new_state = np.concatenate([[init_state[0]], qpos, qvel])
        return env.set_init_state(new_state)

    def transform_task_desc(self, desc, env=None):
        for old, new in self.prompt_replace_mapping:
            desc = desc.replace(old, new)
        return desc

    def manifest(self) -> dict:
        return {
            'kind': 'SceneRemoveObjects',
            'slug': self.slug,
            'remove': list(self.remove),
            'goal_substitutions': [list(p) for p in self.goal_substitutions],
            'prompt_replace_mapping': [list(p) for p in self.prompt_replace_mapping],
            'orig_order': list(self._orig_order),
            'kept_order': list(self._kept_order),
            'derived_bddl_path': self._bddl_path,
        }


## 2. Config

`J_THRESHOLD` caps the number of inferences captured per rollout. With
`num_open_loop_steps=16` and `TASK_MAX_STEPS['libero_10']=600`, a rollout
that runs to step-cap produces ~38 inferences. The first ~8 are typically
common between cluttered and uncluttered rollouts.


In [ ]:
SUITE_NAME   = 'libero_10'
TASK_ID      = 0
N_EPISODES   = 10
J_THRESHOLD  = 8        # capture at most this many inferences per rollout
RESOLUTION   = 256

PROMPT = 'put both the milk and the tomato sauce in the basket'

OUT_DIR = Path('notebooks/lqr/inputs/policy_inputs') / f'{SUITE_NAME}__task{TASK_ID:02d}__milk_first_chunks'
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSITIVE_NPZ  = OUT_DIR / 'positive.npz'
NEGATIVE_NPZ  = OUT_DIR / 'negative.npz'
MANIFEST_JSON = OUT_DIR / 'manifest.json'

POSITIVE_STRESS = SceneRemoveObjects(
    remove=(
        'alphabet_soup_1', 'cream_cheese_1', 'ketchup_1',
        'orange_juice_1', 'butter_1',
    ),
    goal_substitutions=(('alphabet_soup_1', 'milk_1'),),
    prompt_replace_mapping=(('alphabet soup', 'milk'),),
    name_hint='milk_only',
)
NEGATIVE_STRESS = StressTest()

print(f'suite        : {SUITE_NAME}')
print(f'task         : {TASK_ID}')
print(f'episodes     : {N_EPISODES}')
print(f'J_THRESHOLD  : {J_THRESHOLD}  (max inferences captured per rollout)')
print(f'prompt       : {PROMPT!r}')
print(f'positive ->  {POSITIVE_NPZ.resolve()}  (uncluttered = I(u, i, j))')
print(f'negative ->  {NEGATIVE_NPZ.resolve()}  (cluttered   = I(c, i, j))')


## 3. Task suite + init states + model

In [ ]:
task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f'task.language        : {task.language!r}')
print(f'init states available: {init_states.shape[0]}  (using the first {N_EPISODES})')
print(f'max_env_steps        : {max_env_steps}')
assert N_EPISODES <= init_states.shape[0]


In [ ]:
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only',
    ckpt_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B',
    config_file='cosmos_policy/config/config.py',
    dataset_stats_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json',
    t5_text_embeddings_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True,
    use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5,
    num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)
print('model ready')


## 4. Build both envs

Each env stays long-lived across episodes — we `reset()` + `set_init_state()`
per rollout. `env_pos` is the milk-only stripped scene built from
`POSITIVE_STRESS.transform_task`. `env_neg` is the unmodified 8-object scene.


In [ ]:
neg_task = NEGATIVE_STRESS.transform_task(task, output_dir=OUT_DIR)
env_neg, base_task_desc_neg = get_libero_env(neg_task, 'cosmos', resolution=RESOLUTION)
print(f'env_neg={type(env_neg).__name__}  base_task_desc={base_task_desc_neg!r}')

pos_task = POSITIVE_STRESS.transform_task(task, output_dir=OUT_DIR)
env_pos, base_task_desc_pos = get_libero_env(pos_task, 'cosmos', resolution=RESOLUTION)
print(f'env_pos={type(env_pos).__name__}  base_task_desc={base_task_desc_pos!r}')
print(f'orig objects: {list(POSITIVE_STRESS._orig_order)}')
print(f'kept objects: {list(POSITIVE_STRESS._kept_order)}')
print(f'derived BDDL: {POSITIVE_STRESS._bddl_path}')


## 5. Independent first-J-inferences rollout

One env, one rollout, captured at the first `J_THRESHOLD` inference calls.
Stops early if the rollout finishes (success) before `J_THRESHOLD`
inferences. The returned list contains exactly
`min(J_THRESHOLD, num_inferences_before_done)` records.


In [ ]:
def policy_fn(obs, desc):
    out = get_action(
        cfg, model, dataset_stats, obs, desc,
        num_denoising_steps_action=cfg.num_denoising_steps_action,
        generate_future_state_and_value_in_parallel=True,
    )
    return out['actions']


def rollout_collect_first_chunks(
    env, init_state, prompt, stress, *, J, num_steps_wait=10,
):
    """Run a rollout in `env` from `init_state`. Return up to `J` policy-input
    records, one per inference call, in the order they happened.

    `stress.set_init_state(env, init_state)` is used so that env_pos (which
    has fewer bodies than the saved init_state) gets the correctly-sliced
    starting state. For env_neg (StressTest base class) this is a passthrough.

    Returns (success, env_steps, inputs_list).
    """
    env.reset()
    obs = stress.set_init_state(env, init_state)

    # Settle (matches run_libero_eval).
    for _ in range(num_steps_wait):
        obs, _, _, _ = env.step(get_libero_dummy_action(cfg.model_family))

    queue = deque(maxlen=cfg.num_open_loop_steps)
    inputs = []
    success = False
    t = 0
    while t < max_env_steps and len(inputs) < J:
        if not queue:
            observation = prepare_observation(
                obs, resize_size=224, flip_images=cfg.flip_images,
            )
            inputs.append({
                'primary_image': np.ascontiguousarray(observation['primary_image']),
                'wrist_image':   np.ascontiguousarray(observation['wrist_image']),
                'proprio':       np.asarray(observation['proprio'], dtype=np.float32),
            })
            if len(inputs) >= J:
                break  # don't bother computing the next action chunk
            actions = policy_fn(observation, prompt)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs, _, done, _ = env.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, inputs


## 6. Run both envs across all episodes, pair by (i, j)

For each episode `i`, run an independent rollout in `env_pos` and `env_neg`
under the same `PROMPT` and the same `init_states[i]`. Capture up to
`J_THRESHOLD` inference inputs from each. Keep only the row range that both
rollouts reached — that gives us `min(J_pos_i, J_neg_i)` paired rows for
episode `i`.


In [ ]:
all_pos_primary, all_pos_wrist, all_pos_proprio = [], [], []
all_neg_primary, all_neg_wrist, all_neg_proprio = [], [], []
all_episode_idx, all_inference_idx = [], []
rollout_summaries = []

for ep in range(N_EPISODES):
    # --- uncluttered (positive) rollout ---
    t0 = time.time()
    succ_u, steps_u, inputs_u = rollout_collect_first_chunks(
        env_pos, init_states[ep], PROMPT, POSITIVE_STRESS, J=J_THRESHOLD,
    )
    dt_u = time.time() - t0

    # --- cluttered (negative) rollout ---
    t0 = time.time()
    succ_c, steps_c, inputs_c = rollout_collect_first_chunks(
        env_neg, init_states[ep], PROMPT, NEGATIVE_STRESS, J=J_THRESHOLD,
    )
    dt_c = time.time() - t0

    n_pair = min(len(inputs_u), len(inputs_c))
    for j in range(n_pair):
        u_rec = inputs_u[j]
        c_rec = inputs_c[j]
        all_pos_primary.append(u_rec['primary_image'])
        all_pos_wrist.append(u_rec['wrist_image'])
        all_pos_proprio.append(u_rec['proprio'])
        all_neg_primary.append(c_rec['primary_image'])
        all_neg_wrist.append(c_rec['wrist_image'])
        all_neg_proprio.append(c_rec['proprio'])
        all_episode_idx.append(ep)
        all_inference_idx.append(j)

    tag_u = 'SUCCESS' if succ_u else 'FAILURE'
    tag_c = 'SUCCESS' if succ_c else 'FAILURE'
    print(
        f'  ep {ep:2d}  '
        f'pos: {tag_u:7s} J={len(inputs_u):2d}/{J_THRESHOLD} steps={steps_u:4d} ({dt_u:5.1f}s)  '
        f'neg: {tag_c:7s} J={len(inputs_c):2d}/{J_THRESHOLD} steps={steps_c:4d} ({dt_c:5.1f}s)  '
        f'paired={n_pair}'
    )
    rollout_summaries.append({
        'episode': ep,
        'pos': {'success': bool(succ_u), 'env_steps': int(steps_u),
                'n_inferences': len(inputs_u), 'wall_time_s': dt_u},
        'neg': {'success': bool(succ_c), 'env_steps': int(steps_c),
                'n_inferences': len(inputs_c), 'wall_time_s': dt_c},
        'n_paired': n_pair,
    })

env_neg.close()
env_pos.close()
print()
print(f'total paired rows: {len(all_episode_idx)}  '
      f'(theoretical max: {N_EPISODES * J_THRESHOLD})')


## 7. Save paired NPZs

Schema matches the original `collect_policy_inputs_milk` output so the
downstream SVD scripts (`notebooks/lqr/svd/run_partition_svd_pairs.py`) can
read these files unchanged. `drive_source` is a placeholder all-zero column
here (this notebook has no driving distinction) — use `--drive-source all`
(the default) downstream.


In [ ]:
def _stack_save(out_npz, primary_list, wrist_list, proprio_list):
    primary_arr   = np.stack(primary_list, axis=0)
    wrist_arr     = np.stack(wrist_list,   axis=0)
    proprio_arr   = np.stack(proprio_list, axis=0)
    episode_arr   = np.asarray(all_episode_idx,   dtype=np.int32)
    inference_arr = np.asarray(all_inference_idx, dtype=np.int32)
    drive_arr     = np.zeros(len(all_episode_idx), dtype=np.int32)  # placeholder
    np.savez_compressed(
        out_npz,
        primary_images=primary_arr,
        wrist_images=wrist_arr,
        proprios=proprio_arr,
        episode_idx=episode_arr,
        inference_idx=inference_arr,
        drive_source=drive_arr,
    )
    size_mb = out_npz.stat().st_size / 1e6
    print(f'wrote {out_npz}  ({size_mb:.1f} MB)')
    print(f'  primary_images: {primary_arr.shape}  dtype={primary_arr.dtype}')
    print(f'  wrist_images:   {wrist_arr.shape}  dtype={wrist_arr.dtype}')
    print(f'  proprios:       {proprio_arr.shape}  dtype={proprio_arr.dtype}')
    return primary_arr.shape[0]


n_pos = _stack_save(POSITIVE_NPZ, all_pos_primary, all_pos_wrist, all_pos_proprio)
n_neg = _stack_save(NEGATIVE_NPZ, all_neg_primary, all_neg_wrist, all_neg_proprio)
assert n_pos == n_neg, 'paired-row invariant violated after save'

# Proprio is NOT identical across the pair here — quantify how far it drifts.
pos_proprio = np.stack(all_pos_proprio, axis=0)
neg_proprio = np.stack(all_neg_proprio, axis=0)
proprio_max  = float(np.max(np.abs(pos_proprio - neg_proprio)))
proprio_mean = float(np.mean(np.abs(pos_proprio - neg_proprio)))
print(f'paired proprio |diff| max  : {proprio_max:.3e}  (expected: nonzero — independent rollouts)')
print(f'paired proprio |diff| mean : {proprio_mean:.3e}')


## 8. Visualize a few paired examples

Sanity-check that the cluttered and uncluttered images are paired by
`(episode, inference)`. Pick a few inference indices from one episode and
render side-by-side primary and wrist views.


In [ ]:
import matplotlib.pyplot as plt

VIZ_EPISODE  = 0
VIZ_SAMPLES  = min(4, J_THRESHOLD)

episode_arr_all   = np.asarray(all_episode_idx,   dtype=np.int32)
inference_arr_all = np.asarray(all_inference_idx, dtype=np.int32)

mask = episode_arr_all == VIZ_EPISODE
rows_in_ep = np.where(mask)[0]
if len(rows_in_ep) == 0:
    print(f'no rows for episode {VIZ_EPISODE}; nothing to visualize')
else:
    pick = rows_in_ep[np.linspace(0, len(rows_in_ep) - 1, min(VIZ_SAMPLES, len(rows_in_ep))).astype(int)]
    n_rows = len(pick)
    fig, axes = plt.subplots(n_rows, 4, figsize=(13, 3.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for row_i, i in enumerate(pick):
        j = int(inference_arr_all[i])
        dprop = float(np.max(np.abs(all_pos_proprio[i] - all_neg_proprio[i])))
        row_tag = f'ep{VIZ_EPISODE}  j={j:2d}'
        axes[row_i, 0].imshow(all_neg_primary[i])
        axes[row_i, 0].set_title(f'{row_tag}\nneg primary  (cluttered)', fontsize=9)
        axes[row_i, 0].axis('off')
        axes[row_i, 1].imshow(all_pos_primary[i])
        axes[row_i, 1].set_title('pos primary  (uncluttered)', fontsize=9)
        axes[row_i, 1].axis('off')
        axes[row_i, 2].imshow(all_neg_wrist[i])
        axes[row_i, 2].set_title('neg wrist', fontsize=9)
        axes[row_i, 2].axis('off')
        axes[row_i, 3].imshow(all_pos_wrist[i])
        axes[row_i, 3].set_title(f'pos wrist   |Δproprio|∞={dprop:.1e}', fontsize=9)
        axes[row_i, 3].axis('off')
    plt.tight_layout()
    plt.show()


## 9. Manifest

In [ ]:
manifest = {
    'suite': SUITE_NAME,
    'task_id': TASK_ID,
    'n_episodes': N_EPISODES,
    'J_threshold': J_THRESHOLD,
    'resolution': RESOLUTION,
    'prompt': PROMPT,
    'pairing': (
        'row i in positive.npz pairs with row i in negative.npz by '
        '(episode_idx[i], inference_idx[i]) — both share the same prompt and '
        'init_state, but each rollout stepped its own env independently. '
        'Proprio at row i differs across the two files: the policies drift '
        'apart after a few inferences.'
    ),
    'image_layout': 'HWC uint8, flip_images=True applied at capture time (same as get_action input)',
    'proprio_layout': 'concat(robot0_gripper_qpos[2], robot0_eef_pos[3], robot0_eef_quat[4]) -> shape (9,) float32',
    'drive_source_in_npz': 'placeholder all-zero column; this notebook has no driving distinction',
    'sets': {
        'positive': {
            'out_npz': str(POSITIVE_NPZ),
            'stress_test': POSITIVE_STRESS.manifest(),
            'env_base_task_desc': base_task_desc_pos,
            'role': 'uncluttered (milk_only) scene rollout, first J inferences',
        },
        'negative': {
            'out_npz': str(NEGATIVE_NPZ),
            'stress_test': NEGATIVE_STRESS.manifest(),
            'env_base_task_desc': base_task_desc_neg,
            'role': 'cluttered (full 8-object) scene rollout, first J inferences',
        },
    },
    'rollouts': rollout_summaries,
    'paired_proprio_abs_diff': {
        'max': proprio_max,
        'mean': proprio_mean,
    },
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2))
print(f'wrote {MANIFEST_JSON}')


## 10. Summary

In [ ]:
succ_pos = sum(r['pos']['success'] for r in rollout_summaries)
succ_neg = sum(r['neg']['success'] for r in rollout_summaries)
inf_pos  = sum(r['pos']['n_inferences'] for r in rollout_summaries)
inf_neg  = sum(r['neg']['n_inferences'] for r in rollout_summaries)
paired   = sum(r['n_paired'] for r in rollout_summaries)
print(f'  pos (uncluttered): success={succ_pos}/{N_EPISODES}  inferences captured={inf_pos}')
print(f'  neg (cluttered)  : success={succ_neg}/{N_EPISODES}  inferences captured={inf_neg}')
print(f'  paired rows      : {paired}  (max possible: {N_EPISODES * J_THRESHOLD})')
print(f'  paired |Δproprio|: max={proprio_max:.3e}  mean={proprio_mean:.3e}')
print(f'  positive  : {POSITIVE_NPZ.resolve()}')
print(f'  negative  : {NEGATIVE_NPZ.resolve()}')
print(f'  manifest  : {MANIFEST_JSON.resolve()}')
